# Libraries

In [1]:
# Import Packages
import os
import numpy as np
import polars as pl
import pyarrow.dataset as pads

# Establish OF POS

In [2]:
# Read in what pos to process
of_pos_file = open("of_pos.txt", "r")
of_pos = int(of_pos_file.read())
of_pos_file.close()

# Read in Data

In [ ]:
####################################################
### Code Taken from SMT_Data_Starter
####################################################

# For the data_path argument, include the full file path to the folder that holds the data!    
def readDataSubset(table_type, data_path=):
    '''
    Takes a the set of tables from Data set and
    transforms into panda data frame for manipulation
    
    Input:
    -     table_type   = Set of tables of interest from Data set [str]
    -     data_path    = File Path which data is located in Local device, change as necessary[str]
    
    '''
    if table_type not in ['ball-positions', 'ball-events', 'player-positions', 'lineups', 'game-info']:
        print("Invalid data subset name. Please try again with a valid data subset.")
        return -1

    if table_type == 'lineups' or table_type == 'game-info':
        return pads.dataset(source = os.path.join(os.path.dirname(__name__), data_path, table_type + '.csv'), format = 'csv')
    else:
        return pads.dataset(source = os.path.join(os.path.dirname(__name__), data_path, table_type), format = 'csv', partitioning = ['home_team', 'away_team', 'year', 'day'])

In [4]:
####################################################
### Code Taken from SMT_Data_Starter
####################################################

# Read in the Ball Events Data
ball_events_subset = readDataSubset('ball-events')
ball_events = ball_events_subset.to_table().to_pandas()
ball_events_df = pl.from_pandas(ball_events)

# Read in the Ball Pos Data
ball_pos_subset = readDataSubset('ball-positions')
ball_pos = ball_pos_subset.to_table().to_pandas()
ball_pos_df = pl.from_pandas(ball_pos)

# Read in the Player Pos Data for chosen OF pos
player_pos_subset = readDataSubset('player-positions')
player_pos = player_pos_subset.to_table(filter = (pads.field("player_id").isin([of_pos, 10]))).to_pandas()
player_pos_df = pl.from_pandas(player_pos)

# Find Timestamps for Contact and Next EventFind timestamps for contact and next event

In [5]:
# Find the timestamp for contact of all hits that either bounce off the ground or are caught by the targeted fielder
bbip_hit_df = ball_events_df \
                               .filter(
                                   (pl.col("player_id") == 10) &
                                   (pl.col("ball_eventcode") == 4) &
                                   (pl.col("player_id").shift(-1).is_in([of_pos, 255])) &
                                   (pl.col("ball_eventcode").shift(-1).is_in([2, 16]))
                                  ) \
                                .with_columns(
                                    pl.col("timestamp").alias("hit_timestamp")
                                ) \
                                .select(
                                    ["game_string", "play_per_game", "hit_timestamp"]
                                ) \
                                .drop_nulls(
                                ) \
                                .sort(
                                    ["game_string", "play_per_game"]
                                     )

# Find the timestamp after contact of all hits that either bounce off the ground or are caught by the targeted fielder to give us the next event
bbip_next_event_df = ball_events_df \
                               .filter(
                                   (pl.col("player_id").shift(1) == 10) &
                                   (pl.col("ball_eventcode").shift(1) == 4) &
                                   (pl.col("player_id").is_in([of_pos, 255])) &
                                   (pl.col("ball_eventcode").is_in([2, 16]))
                                  ) \
                                .with_columns(
                                    pl.col("timestamp").alias("next_event_timestamp")
                                ) \
                                .select(
                                      ["game_string", "play_per_game", "player_id", "ball_eventcode", "timestamp", "next_event_timestamp"]
                                )\
                                .drop_nulls(
                                ) \
                                .sort(
                                    ["game_string", "play_per_game"]
                                )
# Join above data frames
full_ball_event_df = bbip_next_event_df \
                                .join(
                                    bbip_hit_df,
                                    how = "inner",
                                    on = ["game_string", "play_per_game"]
                                ) \
                                .select(
                                    ["game_string", "play_per_game", "player_id", "ball_eventcode", "timestamp", "hit_timestamp", "next_event_timestamp"]
                                )

# Get Start and End POS of OFer

In [6]:
# Isolate OFer position at contact
hitter_pos_df = player_pos_df \
                              .join( 
                                   full_ball_event_df,
                                   how = "left",
                                   on = ["game_string", "play_per_game"]
                                    ) \
                                .filter(
                                    (pl.col("player_id") == of_pos) &
                                    (pl.col("timestamp") == pl.col("hit_timestamp"))
                                ) \
                                .select(
                                    ["game_string", "play_per_game", "player_id", "field_x", "field_y", "timestamp", "hit_timestamp"]
                                    )
# Isolate ball position at next event
next_event_df = full_ball_event_df \
                                .join( 
                                    ball_pos_df,
                                    how = "inner",
                                    on = ["game_string", "play_per_game", "timestamp"]
                                )\
                                .filter(
                                    (pl.col("timestamp") == pl.col("next_event_timestamp")).over(["game_string", "play_per_game"])
                                )\
                                .select(
                                    ["game_string", "play_per_game", "player_id", "ball_eventcode", "ball_position_x", "ball_position_y", "ball_position_z", "timestamp", "next_event_timestamp"]
                                    )
# Join above data frames
start_end_df = next_event_df \
                                .join(
                                     hitter_pos_df,
                                     how = "inner",
                                     on = ["game_string", "play_per_game"] 
                                    ) \
                                .select(["game_string", "play_per_game", "player_id", "ball_eventcode", "ball_position_x",
                                         "ball_position_y", "ball_position_z", "timestamp", "hit_timestamp","next_event_timestamp"]
                                ) \
                                .sort(
                                    ["game_string", "play_per_game", "timestamp"]
                                )

# Infer Handedness of Batter and Calculate Fielder Starting Location

In [7]:
# define lefty and righty based on their position at time of contact relative to homeplate
splits_df = hitter_pos_df\
                        .select(["game_string", "play_per_game", "field_x", "hit_timestamp"]
                               )\
                        .with_columns( # https://docs.pola.rs/api/python/stable/reference/expressions/api/polars.when.html
                            pl.when(pl.col("field_x") > 0).then(1).otherwise(2).alias("bats") # 1 is lefty, 2 is righty
                                )
# join splits with player pos to define target ofer starting location
start_pos_df = splits_df \
                        .join(
                            player_pos_df,
                            how = "left",
                            on = ["game_string", "play_per_game"]
                        )\
                        .filter(
                            (pl.col("player_id") == of_pos) &
                            (pl.col("timestamp") == pl.col("hit_timestamp"))                            
                        )\
                        .select(
                           ["game_string", "play_per_game", "player_id", "hit_timestamp", "bats",
                            pl.col("field_x").alias("start_x"),
                            pl.col("field_y").alias("start_y")
                           ]
                        ) \
                        .sort(
                           ["game_string", "play_per_game"]
                        )
# define average versus right handed hitters
start_pos_vr = start_pos_df \
                        .filter(
                            pl.col("bats") == 2
                        ) \
                        .select(
                            ["start_x", "start_y"]
                        )

avg_start_x_vr = start_pos_vr.get_column("start_x").mean()
avg_start_y_vr = start_pos_vr.get_column("start_y").mean()

delta_pos_vr = start_pos_vr \
                        .with_columns(
                            (pl.col("start_x") - avg_start_x_vr).alias("delta_x"),
                            (pl.col("start_y") - avg_start_y_vr).alias("delta_y")
                        )
# define average versus left handed hitters
start_pos_vl = start_pos_df \
                        .filter(
                            pl.col("bats") == 1
                        ) \
                        .select(
                            ["start_x", "start_y"]
                        )

avg_start_x_vl = start_pos_vl.get_column("start_x").mean()
avg_start_y_vl = start_pos_vl.get_column("start_y").mean()

delta_pos_vl = start_pos_vl \
                        .with_columns(
                            (pl.col("start_x") - avg_start_x_vl).alias("delta_x"),
                            (pl.col("start_y") - avg_start_y_vl).alias("delta_y")
                        )

# See How Much Distance Was Gained or Lost by Shift and Calculate Attack Angle from Avg Positon

In [8]:
# join with timestamp dataframe to calculate distance
dist_split_df = start_end_df \
                        .join(
                            start_pos_df,
                            how = "inner",
                            on = ["game_string", "play_per_game"]
                        ) \
                        .select(
                            ["game_string", "play_per_game", "player_id", "ball_eventcode", "next_event_timestamp", "ball_position_x", "ball_position_y", "start_x", "start_y", "bats"]
                        ) \
                        .sort(
                            ["game_string", "play_per_game"]
                        ) \
                        .with_columns( # actual distance
                            ((pl.col("ball_position_y") - pl.col("start_y")) ** 2 + ((pl.col("ball_position_x") - pl.col("start_x")) ** 2)).sqrt() \
                            .alias("actual_dist").over(["game_string", "play_per_game"]),
                            # calculate distance based on average starting location by batter split
                            (((pl.col("ball_position_y") - pl.when(pl.col("bats") == 1) .then(avg_start_y_vl) .otherwise(avg_start_y_vr)) ** 2 + \
                            ((pl.col("ball_position_x") - pl.when(pl.col("bats") == 1) .then(avg_start_x_vl) .otherwise(avg_start_x_vr))) ** 2) \
                            .sqrt().alias("avg_dist_split")).over(["game_string", "play_per_game"]),
                            # calculate attack angle based on average starting location by batter split
                            (pl.arctan2((pl.when(pl.col("bats") == 1).then(avg_start_y_vl).otherwise(avg_start_y_vr) - (pl.col("ball_position_y"))),
                            (pl.when(pl.col("bats") == 1).then(avg_start_x_vl).otherwise(avg_start_x_vr) - (pl.col("ball_position_x")))) \
                            .degrees()).alias("attack_angle_avg_deg").over(["game_string", "play_per_game"])
                        )
# calculate distance delta
dist_delta_df = dist_split_df \
                            .with_columns(
                                (pl.col("avg_dist_split") - pl.col("actual_dist")).alias("dist_delta")
                            ) \
                            .sort(
                                pl.col("dist_delta").abs(),
                                descending = True
                            )